In [0]:
dbutils.widgets.text("p_environment", "production")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../Includes/configuration"

In [0]:
%run "../Includes/common_functions"

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

In [0]:
movie_languages_schema = StructType(fields= [
                         StructField("movieId", IntegerType(), True),
                         StructField("languageId", IntegerType(), True),
                         StructField("languageRoleId", IntegerType(), True)
])

In [0]:
movie_languages_df = spark.read \
                     .schema(movie_languages_schema) \
                     .option("multiline", True) \
                     .json(f"{bronze_folder_path}/{v_file_date}/movie_language")

In [0]:
display(movie_languages_df)

movieId,languageId,languageRoleId
33644,24574,1
1649,24574,1
9895,24574,1
9570,24574,1
27579,24574,1
16052,24574,1
40264,24574,1
1164,24574,1
239678,24574,1
14359,24574,1


In [0]:
movie_languages_df.count()

2730

In [0]:
from pyspark.sql.functions import col

In [0]:
movie_languages_dropped_df = movie_languages_df.drop(col("languageRoleId"))

In [0]:
display(movie_languages_dropped_df)

movieId,languageId
33644,24574
1649,24574
9895,24574
9570,24574
27579,24574
16052,24574
40264,24574
1164,24574
239678,24574
14359,24574


In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
movie_languages_final_df = add_ingestion_date(movie_languages_dropped_df) \
                            .withColumnsRenamed({"movieId": "movie_Id", "languageId": "language_Id"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))

In [0]:
display(movie_languages_final_df)

movie_Id,language_Id,ingestion_date,environment,file_date
33644,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
1649,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
9895,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
9570,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
27579,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
16052,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
40264,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
1164,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
239678,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30
14359,24574,2026-09-11T04:40:08.862723Z,production,2024-12-30


In [0]:
# overwrite_partition("movie_silver", "movie_languages", "file_date", v_file_date)

In [0]:
merge_delta_lake_2(movie_languages_final_df, "movie_silver", "movie_languages", "movie_Id", "language_Id", "file_date")

In [0]:
# movie_languages_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movie_languages")

In [0]:
display(spark.read.table("movie_silver.movie_languages"))

movie_Id,language_Id,ingestion_date,environment,file_date
36593,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36597,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36643,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36647,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36648,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36657,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36658,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36668,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36669,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16
36670,24574,2026-09-11T04:35:11.150641Z,production,2024-12-16


In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movie_languages
GROUP BY file_date;

file_date,count(1)
2024-12-16,6000
2024-12-23,3010
2024-12-30,2730


In [0]:
%sql
DESCRIBE EXTENDED movie_silver.movie_languages;

col_name,data_type,comment
movie_Id,int,null
language_Id,int,null
ingestion_date,timestamp,null
environment,string,null
file_date,string,null
# Partition Information,,
# col_name,data_type,comment
file_date,string,null
,,
# Delta Statistics Columns,,


In [0]:
dbutils.notebook.exit("Success")